#1)  import libraries

In [100]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import tensorflow as tf
import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
# tf.keras.utils.set_random_seed(42)

#2) Get data

In [2]:
data=pd.read_csv('/content/bengaluru_house_prices.csv')
data.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
data.shape


(13320, 9)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [5]:
data.describe()

,bath,balcony,price
count,13247.000000,12711.000000,13320.000000
mean,2.692610,1.584376,112.565627
std,1.341458,0.817263,148.971674
min,1.000000,0.000000,8.000000
25%,2.000000,1.000000,50.000000
50%,2.000000,2.000000,72.000000
75%,3.000000,2.000000,120.000000
max,40.000000,3.000000,3600.000000


In [6]:
data.duplicated().sum()

np.int64(529)

In [7]:
data.drop_duplicates(inplace=True)

In [8]:
data.shape

(12791, 9)

In [9]:
data.isna().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5328
total_sqft,0
bath,73
balcony,605
price,0


#3) Data preprocessing

###3.1) Nulls

In [10]:
# data[data['size'].isna()]
# Show rows where size, bath and balcony are all missing

data[data[['size', 'bath', 'balcony']].isna().all(axis=1)]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
579,Plot Area,Immediate Possession,Sarjapur Road,NaN,Asiss B,1200 - 2400,NaN,NaN,34.185
1775,Plot Area,Immediate Possession,IVC Road,NaN,Orana N,2000 - 5634,NaN,NaN,124.000
2264,Plot Area,Immediate Possession,Banashankari,NaN,NaN,2400,NaN,NaN,460.000
2809,Plot Area,Immediate Possession,Sarjapur Road,NaN,AsdiaAr,1200 - 2400,NaN,NaN,28.785
2862,Plot Area,Immediate Possession,Devanahalli,NaN,Ajleyor,1500 - 2400,NaN,NaN,46.800
5333,Plot Area,Immediate Possession,Devanahalli,NaN,Emngs S,2100 - 5405,NaN,NaN,177.115
6423,Plot Area,Immediate Possession,Whitefield,NaN,SRniaGa,2324,NaN,NaN,26.730
6636,Plot Area,Immediate Possession,Jigani,NaN,S2enste,1500,NaN,NaN,25.490
6719,Plot Area,Immediate Possession,Hoskote,NaN,SJowsn,800 - 2660,NaN,NaN,28.545
7680,Plot Area,Immediate Possession,Kasavanhalli,NaN,NaN,5000,NaN,NaN,400.000


In [11]:
# Delete those rows
data = data.dropna(subset=['size', 'bath', 'balcony'], how='all')

In [12]:
data[data[['size', 'bath', 'balcony']].isna().all(axis=1)]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price


In [13]:
data[data['location'].isna()]

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
568,Super built-up Area,Ready To Move,NaN,3 BHK,Grare S,1600,3.0,2.0,86.0


In [14]:
data = data.dropna(subset=['location'])

In [15]:
data = data.drop('society', axis=1)     # we have approximately 40 % nulls and column contain diffrent values, so drop it for NN

In [16]:
data.isna().sum()

,0
area_type,0
availability,0
location,0
size,0
total_sqft,0
bath,57
balcony,589
price,0


In [17]:
data['bath'] = data['bath'].fillna(data['bath'].median())      # why medain? bath might have outlier values, and the median is safer than the mean

In [18]:
data['balcony'] = data['balcony'].fillna(data['balcony'].median())

In [19]:
data.isna().sum()

,0
area_type,0
availability,0
location,0
size,0
total_sqft,0
bath,0
balcony,0
price,0


###3.2) outliers

In [20]:
Q1 = data["price"].quantile(0.25)
Q3 = data["price"].quantile(0.75)
lower_bound = Q1 - 1.5 * (Q3 - Q1)
upper_bound = Q3 + 1.5 * (Q3 - Q1)
outliers = data[(data["price"] < lower_bound) | (data["price"] > upper_bound)]
outliers
outliers.shape

(1255, 8)

In [21]:
print("Q1 =", Q1)
print("Q3 =", Q3)
print("Lower =", lower_bound)
print("Upper =", upper_bound)
print("Min =", data["price"].min())
print("Max =", data["price"].max())

Q1 = 50.0
Q3 = 121.0
Lower = -56.5
Upper = 227.5
Min = 8.0
Max = 3600.0


In [22]:
data["price"].describe()         # price distribution واضح إنه skewed جدا

,price
count,12774.000000
mean,114.339147
std,151.506067
min,8.000000
25%,50.000000
50%,73.000000
75%,121.000000
max,3600.000000


###3.3) Handling Text

In [23]:
print("area_type:", data['area_type'].nunique())
print("availability:", data['availability'].nunique())
print("location:", data['location'].nunique())

area_type: 4
availability: 80
location: 1304


In [49]:
pd.get_dummies(data, columns=['area_type'], dtype=int)    # Area Type

,availability,location,total_sqft,bath,balcony,price,bhk,area_type_Built-up Area,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,0,Electronic City Phase II,1056.0,2.0,1.0,39.07,2.0,0,0,0,1
1,1,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4.0,0,0,1,0
2,1,Uttarahalli,1440.0,2.0,3.0,62.00,3.0,1,0,0,0
3,1,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3.0,0,0,0,1
4,1,Kothanur,1200.0,2.0,1.0,51.00,2.0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
13314,1,Green Glen Layout,1715.0,3.0,3.0,112.00,3.0,0,0,0,1
13315,1,Whitefield,3453.0,4.0,0.0,231.00,5.0,1,0,0,0
13316,1,other,3600.0,5.0,2.0,400.00,4.0,0,0,0,1
13317,1,Raja Rajeshwari Nagar,1141.0,2.0,1.0,60.00,2.0,1,0,0,0


In [25]:
data['location'] = data['location'].apply(lambda x: str(x).strip())    # delete space

location_stats = data.groupby('location')['location'].agg('count')     # show repeation time

locations_less_than_10 = location_stats[location_stats <= 10]          # if area repeated at least 10 model cosider it ,otherwise cant learn it

data['location'] = data['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)

print("عدد الأماكن الفريدة بعد التجميع:", data['location'].nunique())

عدد الأماكن الفريدة بعد التجميع: 234


In [65]:
data = pd.get_dummies(data, columns=['location', 'area_type'], drop_first=True, dtype=float)

In [26]:
# 1-- ready ,2-- unready or under construction
data['availability'] = data['availability'].apply(lambda x: 1 if str(x).strip() == 'Ready To Move' else 0)

print("new distrbution values : ")
print(data['availability'].value_counts())

new distrbution values : 
availability
1    10171
0     2603
Name: count, dtype: int64


In [27]:
data_encoded = pd.get_dummies(data, columns=['area_type', 'location'], drop_first=True, dtype=float)

print("final shape : ",data_encoded.shape)

final shape :  (12774, 242)


In [28]:
data['bhk'] = data['size'].str.extract(r'(\d+)').astype(float)   # bhk : Bedroom, Hall, Kitchen

In [29]:
data[['size', 'bhk']].head(10)

,size,bhk
0,2 BHK,2.0
1,4 Bedroom,4.0
2,3 BHK,3.0
3,3 BHK,3.0
4,2 BHK,2.0
5,2 BHK,2.0
6,4 BHK,4.0
7,4 BHK,4.0
8,3 BHK,3.0
9,6 Bedroom,6.0


In [30]:
data['bhk'].isna().sum()

np.int64(0)

In [31]:
data = data.drop('size', axis=1)

In [32]:
data.head()

,area_type,availability,location,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,0,Electronic City Phase II,1056,2.0,1.0,39.07,2.0
1,Plot Area,1,Chikka Tirupathi,2600,5.0,3.0,120.00,4.0
2,Built-up Area,1,Uttarahalli,1440,2.0,3.0,62.00,3.0
3,Super built-up Area,1,Lingadheeranahalli,1521,3.0,1.0,95.00,3.0
4,Super built-up Area,1,Kothanur,1200,2.0,1.0,51.00,2.0


In [33]:
data['total_sqft'].apply(type).value_counts()     # total_sqft: square feet

,count
total_sqft,
<class 'str'>,12774


In [34]:
data[data['total_sqft'].astype(str).str.contains('-', na=False)]['total_sqft'].head(10)

,total_sqft
30,2100 - 2850
56,3010 - 3410
81,2957 - 3450
122,3067 - 8156
137,1042 - 1105
165,1145 - 1340
188,1015 - 1540
224,1520 - 1740
549,1195 - 1440
661,1120 - 1145


In [35]:
#data['total_sqft'].unique()[:50]

data[
~data['total_sqft'].astype(str).str.match(r'^\d+(\.\d+)?$')
]['total_sqft'].unique()[:50]

     # so total_sqft divide into 3 cat: [    number    ,  range    ,  diffrent measure unit]
                                      #         |           |                 |
                                      #   conv to float , take mean ,    replace to sqft

array(['2100 - 2850', '3010 - 3410', '2957 - 3450', '3067 - 8156',
       '1042 - 1105', '1145 - 1340', '1015 - 1540', '1520 - 1740',
       '34.46Sq. Meter', '1195 - 1440', '4125Perch', '1120 - 1145',
       '4400 - 6640', '3090 - 5002', '4400 - 6800', '1160 - 1195',
       '1000Sq. Meter', '4000 - 5249', '1115 - 1130', '1100Sq. Yards',
       '520 - 645', '1000 - 1285', '3606 - 5091', '650 - 665',
       '633 - 666', '5.31Acres', '30Acres', '1445 - 1455', '884 - 1116',
       '850 - 1093', '1440 - 1884', '716Sq. Meter', '547.34 - 827.31',
       '580 - 650', '3425 - 3435', '1804 - 2273', '3630 - 3800',
       '660 - 670', '1500Sq. Meter', '620 - 933', '142.61Sq. Meter',
       '2695 - 2940', '1574Sq. Yards', '3450 - 3472', '1250 - 1305',
       '670 - 980', '1005.03 - 1252.49', '1004 - 1204', '361.33Sq. Yards',
       '645 - 936'], dtype=object)

In [36]:
  # for detect  diffrent measure units
# data[
#     data['total_sqft'].astype(str).str.contains(
#         'Sq', case=False, na=False
#     )
# ]['total_sqft'].unique()

In [37]:
import re

def convert_sqft(x):

    x = str(x).strip()

    # Case 1: Range
    if '-' in x:
        values = x.split('-')
        return (float(values[0]) + float(values[1])) / 2

    # Case 2: Square Meter
    elif 'Sq. Meter' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 10.7639

    # Case 3: Square Yards
    elif 'Sq. Yards' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 9

    # Case 4: Acres
    elif 'Acres' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 43560

    # Case 5: Perch
    elif 'Perch' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 272.25

    # Case 6: Cents
    elif 'Cents' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 435.6

    # Case 7: Guntha
    elif 'Guntha' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 1089

    # Case 8: Grounds
    elif 'Grounds' in x:
        value = float(re.findall(r'\d+\.?\d*', x)[0])
        return value * 2400

    # Case 9: Normal number
    else:
        return float(x)

In [38]:
data['total_sqft'] = data['total_sqft'].apply(convert_sqft)

In [58]:
# حساب المساحة المخصصة لكل غرفة
data['sqft_per_bhk'] = data['total_sqft'] / data['bhk']

# الاحتفاظ بالشقق اللي مساحة الغرفة فيها >= 300 قدم مربع
data = data[data['sqft_per_bhk'] >= 300]

In [60]:
# 1. إضافة عمود السعر لكل قدم مربع (ملاحظة: السعر بالداتا بالـ Lakhs يعني * 100000)
data['price_per_sqft'] = (data['price'] * 100000) / data['total_sqft']

# 2. فلترة القيم الشاذة في price_per_sqft لكل منطقة (Location)
def remove_pps_outliers(data):
    data_out = pd.DataFrame()
    for key, subdf in data.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        # الاحتفاظ فقط بالقيم التي تقع في حدود Standard Deviation واحد (Mean ± 1*Std)
        reduced_data = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        data_out = pd.concat([data_out, reduced_data], ignore_index=True)
    return data_out

df = remove_pps_outliers(data)

In [62]:
# حذف الشقق التي بها عدد حمامات غير منطقي
df = df[df.bath < (df.bhk + 2)]

In [63]:
data['total_sqft'].describe()

,total_sqft
count,1.203600e+04
mean,1.995001e+03
std,1.817245e+04
min,3.000000e+02
25%,1.116000e+03
50%,1.306000e+03
75%,1.718000e+03
max,1.306800e+06


In [47]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12774 entries, 0 to 13318
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     12774 non-null  object 
 1   availability  12774 non-null  int64  
 2   location      12774 non-null  object 
 3   total_sqft    12774 non-null  float64
 4   bath          12774 non-null  float64
 5   balcony       12774 non-null  float64
 6   price         12774 non-null  float64
 7   bhk           12774 non-null  float64
dtypes: float64(5), int64(1), object(2)
memory usage: 898.2+ KB


3.4) Data Splitting

In [72]:
X = data.drop(['price'],axis = 1)
y =data['price']

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=44, shuffle =True)

print('X_train shape is ' , X_train.shape)
print('X_test shape is ' , X_test.shape)
print('y_train shape is ' , y_train.shape)
print('y_test shape is ' , y_test.shape)

X_train shape is  (9027, 243)
X_test shape is  (3009, 243)
y_train shape is  (9027,)
y_test shape is  (3009,)


In [74]:
data.shape

(12036, 244)

In [66]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12036 entries, 0 to 13318
Columns: 244 entries, availability to area_type_Super built-up  Area
dtypes: float64(243), int64(1)
memory usage: 22.5 MB


# 4) NN

In [79]:
K_Model = keras.models.Sequential([
            keras.layers.Input(shape=(X_train.shape[1],)),
            keras.layers.Dense(8,  activation = 'relu'),
             #keras.layers.Dropout(0.1),                          # after trying we need to prevent overfit
            keras.layers.Dense(256, activation = 'relu'),
             #keras.layers.Dropout(0.3),
            keras.layers.Dense(64, activation = 'relu'),
            keras.layers.Dense(32, activation = 'relu'),
             #keras.layers.Dropout(0.2),
            keras.layers.Dense(1, activation = 'relu')
            ])

In [89]:
from keras.optimizers import Adam
K_Model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='mse',
    metrics=['mae']
)

In [90]:
K_Model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 8)              │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 256)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,817 (89.13 KB)

 Trainable params: 22,817 (89.13 KB)

 Non-trainable params: 0 (0.00 B)

In [91]:
history = K_Model.fit(X_train,y_train,
                         validation_data=(X_test,y_test),
                         epochs=100,
                         batch_size=32,
                         verbose=1,
                         callbacks=[tf.keras.callbacks.EarlyStopping(
                                            patience=10,
                                            mode='min',
                                            monitor="val_loss",
                                            restore_best_weights=True)])

Epoch 1/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1601.9333 - mae: 14.2578 - val_loss: 2347.5261 - val_mae: 12.9995
Epoch 2/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1460.5735 - mae: 13.1062 - val_loss: 2615.1008 - val_mae: 13.2893
Epoch 3/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1400.6294 - mae: 12.3965 - val_loss: 2199.8843 - val_mae: 12.2355
Epoch 4/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1298.3800 - mae: 11.2595 - val_loss: 81393.0469 - val_mae: 19.6869
Epoch 5/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6892.2764 - mae: 11.8856 - val_loss: 2218.0571 - val_mae: 10.9176
Epoch 6/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1254.1343 - mae: 10.9413 - val_loss: 2443.1543 - val_mae: 12.6822
Epoch 7/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1196.7157 - mae: 10.5914 - val_loss: 2216.6633 - val_mae: 10.6410
Epoch 8/100
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1137.8068 - mae: 11.0071 - val_loss: 2575.9614 - 

In [101]:
y_pred = K_Model.predict(X_test)

# Calculate R2 Score
score = r2_score(y_test, y_pred)
print(f"R2 Score: {score * 100:.2f}%")

95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
R2 Score: 93.30%


In [103]:
y_mean = np.mean(y_test)
mae_val = mean_absolute_error(y_test, y_pred)  # best MAE

error_percentage = (mae_val / y_mean) * 100
accuracy_percentage = 100 - error_percentage

print(f"Target Avarage : {y_mean:.2f}")
print(f"Error precentage {error_percentage:.2f}%")
print(f"Accuracy precentage {accuracy_percentage:.2f}%")

Target Avarage : 114.28
Error precentage 8.12%
Accuracy precentage 91.88%


In [104]:
K_Model.save('K_Model.keras')

In [106]:
print(os.path.exists('K_Model.keras'))    # for sure

True
